In [0]:
# ============================================================
# 04_category_mapping.ipynb
# Mục tiêu:
# - Join dim_content to most_search_t6 / most_search_t7
# - Build final customer_360_final table
# - Compute Type and Group
# ============================================================

from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
# CONFIG
CATALOG = "workspace"
SCHEMA = "customer360"
DB = f"{CATALOG}.{SCHEMA}"

TABLE_MOST_SEARCH_T6 = f"{DB}.new_most_search_t6"
TABLE_MOST_SEARCH_T7 = f"{DB}.new_most_search_t7"
TABLE_DIM_CONTENT = f"{DB}.dim_content"
TABLE_FINAL = f"{DB}.customer_360_final"

In [0]:
# LOAD TABLES
most_search_t6 = spark.table(TABLE_MOST_SEARCH_T6)
most_search_t7 = spark.table(TABLE_MOST_SEARCH_T7)
dim = spark.table(TABLE_DIM_CONTENT)

In [0]:
# JOIN CATEGORY FOR T6
window = Window.orderBy("user_id")

t6_with_cat = (
    most_search_t6
    .join(
        dim.select(
            col("content"),
            col("category").alias("category_t6")
        ),
        most_search_t6["Most_Search"] == dim["content"],
        "left"
    )
    .fillna("Other", subset=["category_t6"])
    .select(
        col("user_id"),
        col("Most_Search").alias("most_search_t6"),
        col("category_t6")
    )
    # Vì mục đích demo nên chỉ lấy những record có category khác "Other" để đảm bảo chất lượng data, trong thực tế có thể giữ lại tất cả.
    # Thế nên đổi lại luôn user_id theo thứ tự 2 tháng cho giống nhau để dễ join so sánh
    .filter(col("category_t6") != "Other")
    .withColumn(
            "user_id_new",
            lpad(row_number().over(window).cast("string"), 6, "0")
        )
        .drop("user_id")
        .withColumnRenamed("user_id_new", "user_id")
    .limit(272)
)
print(f"Loaded {t6_with_cat.count()} records")
t6_with_cat.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Loaded 272 records
+--------------------+-----------+-------+
|      most_search_t6|category_t6|user_id|
+--------------------+-----------+-------+
|            Việt Nam|      Sport| 000001|
|hàng xóm của tôi ...|    C-Drama| 000002|
|   trực tiếp bóng đá|      Sport| 000003|
|    bong da viet nam|      Sport| 000004|
|   yêu từng centimet|    C-Drama| 000005|
|   trực tiếp bóng đá|      Sport| 000006|
|     bằng chứng thép|    C-Drama| 000007|
|            Việt Nam|      Sport| 000008|
| xem bóng dá vietnam|      Sport| 000009|
|            Việt Nam|      Sport| 000010|
|    làm lại cuộc đời|    C-Drama| 000011|
|        street dance|      Music| 000012|
|             bóng đá|      Sport| 000013|
|                 vtv|       News| 000014|
|việt nam trực tiế...|      Sport| 000015|
|   trực tiếp bóng đá|      Sport| 000016|
|  kamen rider amazon|  Animation| 000017|
|giao hữu việt nam và|      Sport| 000018|
|tôi thấy hoa vàng...|    C-Drama| 000019|
|gia đình chân to ...|    V-Drama| 

In [0]:
# JOIN CATEGORY FOR T7
window = Window.orderBy("user_id")

t7_with_cat = (
    most_search_t7
    .join(
        dim.select(
            col("content"),
            col("category").alias("category_t7")
        ),
        most_search_t7["Most_Search"] == dim["content"],
        "left"
    )
    .fillna("Other", subset=["category_t7"])
    .select(
        col("user_id"),
        col("Most_Search").alias("most_search_t7"),
        col("category_t7")
    )
    .filter(col("category_t7") != "Other")
    .withColumn(
            "user_id_new",
            lpad(row_number().over(window).cast("string"), 6, "0")
        )
        .drop("user_id")
        .withColumnRenamed("user_id_new", "user_id")
    .limit(272)
)
print(f"Loaded {t7_with_cat.count()} records")
t7_with_cat.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Loaded 272 records
+--------------------+------------+-------+
|      most_search_t7| category_t7|user_id|
+--------------------+------------+-------+
|fpt play sở hữu b...|       Sport| 000001|
|             aikatsu|   Animation| 000002|
|vao ma gioi roi d...|   Animation| 000003|
|            thái lầm|  Thai-Drama| 000004|
|     the transporter|      Action| 000005|
|cố tiên sinh, hóa...|     C-Drama| 000006|
|    sword art online|   Animation| 000007|
|          produce 48|Reality Show| 000008|
|đá bong trực tiếp...|       Sport| 000009|
|nữtửbị éqđi laychòng|     C-Drama| 000010|
|fpt play sở hữu b...|       Sport| 000011|
|u19 viet nam u19 ...|       Sport| 000012|
|trực tiếp bóng đá...|       Sport| 000013|
|fairy tail (seaso...|   Animation| 000014|
|                 u23|       Sport| 000015|
|        black clover|   Animation| 000016|
|liên minh công lý...|      Action| 000017|
|trực tiếp u19 việ...|       Sport| 000018|
|       u 19 việt nam|       Sport| 000019|
|            

In [0]:
# JOIN 2 MONTHS
final = (
    t6_with_cat
    .join(t7_with_cat, on="user_id", how="inner") #
)
print(f"Loaded {final.count()} records")
final.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------+--------------------+-----------+--------------------+------------+
|user_id|      most_search_t6|category_t6|      most_search_t7| category_t7|
+-------+--------------------+-----------+--------------------+------------+
| 000001|            Việt Nam|      Sport|fpt play sở hữu b...|       Sport|
| 000002|hàng xóm của tôi ...|    C-Drama|             aikatsu|   Animation|
| 000003|   trực tiếp bóng đá|      Sport|vao ma gioi roi d...|   Animation|
| 000004|    bong da viet nam|      Sport|            thái lầm|  Thai-Drama|
| 000005|   yêu từng centimet|    C-Drama|     the transporter|      Action|
| 000006|   trực tiếp bóng đá|      Sport|cố tiên sinh, hóa...|     C-Drama|
| 000007|     bằng chứng thép|    C-Drama|    sword art online|   Animation|
| 000008|            Việt Nam|      Sport|          produce 48|Reality Show|
| 000009| xem bóng dá vietnam|      Sport|đá bong trực tiếp...|       Sport|
| 000010|            Việt Nam|      Sport|nữtửbị éqđi laychòng|     C-Drama|

In [0]:
# COMPUTE TYPE / GROUP
final = (
    final
    .withColumn(
        "Type",
        when(col("category_t6") == col("category_t7"), "Same")
        .otherwise("Changed")
    )
    .withColumn(
        "Group",
        when(col("Type") == "Same", col("category_t6"))
        .otherwise(concat_ws(" ", col("category_t6"), col("category_t7")))
    )
    .select(
        "user_id",
        "most_search_t6",
        "most_search_t7",
        "category_t6",
        "category_t7",
        "Type",
        "Group"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# SAVE FINAL TABLE
final.write.mode("overwrite").saveAsTable(TABLE_FINAL)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# Checking again
print("customer_360_final rows:", spark.table(TABLE_FINAL).count())

print("\nSample output:")
spark.table(TABLE_FINAL).show(20, truncate=False)

print("\nType distribution:")
spark.table(TABLE_FINAL).groupBy("Type").count().show()

print("\nTop Group distribution:")
spark.table(TABLE_FINAL).groupBy("Group").count().orderBy(col("count").desc()).show(20, truncate=False)

customer_360_final rows: 272

Sample output:
+-------+-------------------------------+--------------------------------------------------------------------------+-----------+------------+-------+------------------+
|user_id|most_search_t6                 |most_search_t7                                                            |category_t6|category_t7 |Type   |Group             |
+-------+-------------------------------+--------------------------------------------------------------------------+-----------+------------+-------+------------------+
|000001 |Việt Nam                       |fpt play sở hữu bản quyền u19 đông nam á 2022 - bản tin bóng đá việt 01/07|Sport      |Sport       |Same   |Sport             |
|000002 |hàng xóm của tôi không chịu lớn|aikatsu                                                                   |C-Drama    |Animation   |Changed|C-Drama Animation |
|000003 |trực tiếp bóng đá              |vao ma gioi roi day! iruma_                                          

In [0]:
EXPORT_BASE = "/Volumes/workspace/customer360/c360_volume"
PATH_EXPORT_C360 = f"{EXPORT_BASE}/customer_360_final"
final.write.mode("overwrite").parquet(PATH_EXPORT_C360)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
